##1.1股票下载部分

In [1]:
import baostock as bs
import pandas as pd
from datetime import datetime
import time
import os

In [3]:
# ========== 1.1 自选10只股票下载 ==========
print("="*60)
print("1.1 下载股票数据")
print("="*60)

# 创建新的文件夹结构
data_folder = "data"
stock_folder = os.path.join(data_folder, "stock")
if not os.path.exists(stock_folder):
    os.makedirs(stock_folder)
    print(f"✓ 创建文件夹: {stock_folder}/")

# 日志文件
log_file = "download_log.txt"

def write_log(status, data_type, name, details=""):
    """写入下载日志"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if status == "SUCCESS":
        log_entry = f"[{timestamp}] SUCCESS  {data_type}_{name} {details}\n"
    else:
        log_entry = f"[{timestamp}] FAILED   {data_type}_{name} Error: {details}\n"
    
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(log_entry)
    print(log_entry.strip())

# 清空或创建日志文件
with open(log_file, "w", encoding="utf-8") as f:
    f.write(f"# 数据下载日志\n# 开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

# 股票列表
stocks = [
    ("600036", "招商银行", "银行"),
    ("601166", "兴业银行", "银行"),
    ("000625", "长安汽车", "汽车"),
    ("601633", "长城汽车", "汽车"),
    ("000002", "万科A", "房地产"),
    ("600519", "贵州茅台", "白酒"),
    ("000858", "五粮液", "白酒"),
    ("601857", "中国石油", "能源"),
    ("600050", "中国联通", "通讯"),
    ("002352", "顺丰控股", "物流"),
]

def convert_code(code):
    if code.startswith('6'):
        return f"sh.{code}"
    else:
        return f"sz.{code}"

start_date = "2020-01-01"
end_date = datetime.now().strftime("%Y-%m-%d")

# 登录 baostock
bs.login()

# 下载股票数据
for code, name, industry in stocks:
    try:
        print(f"正在下载: {code} {name} ({industry})")
        bs_code = convert_code(code)
        
        rs = bs.query_history_k_data_plus(
            bs_code,
            "date,open,high,low,close,volume,amount",
            start_date=start_date,
            end_date=end_date,
            frequency="d",
            adjustflag="1"
        )
        
        data_list = []
        while (rs.error_code == '0') and rs.next():
            data_list.append(rs.get_row_data())
        
        if data_list:
            df = pd.DataFrame(data_list, 
                            columns=['日期', '开盘', '最高', '最低', '收盘', '成交量', '成交额'])
            for col in ['开盘', '最高', '最低', '收盘', '成交量', '成交额']:
                df[col] = pd.to_numeric(df[col])
            
            # 新文件名格式：stock_代码_名称.csv
            filename = os.path.join(stock_folder, f"stock_{code}.csv")
            df.to_csv(filename, index=False, encoding="utf-8-sig")
            
            write_log("SUCCESS", "stock", code, f"shape={df.shape}")
            print(f"  ✓ 成功！共 {len(df)} 条记录")
        else:
            raise Exception("No data returned")
        
        time.sleep(0.3)
        
    except Exception as e:
        write_log("FAILED", "stock", code, str(e))
        print(f"  ✗ 失败: {e}")

bs.logout()
print("\n股票数据下载完成！")

1.1 下载股票数据
login success!
正在下载: 600036 招商银行 (银行)
[2026-04-10 10:41:50] SUCCESS  stock_600036 shape=(1517, 7)
  ✓ 成功！共 1517 条记录
正在下载: 601166 兴业银行 (银行)
[2026-04-10 10:41:53] SUCCESS  stock_601166 shape=(1517, 7)
  ✓ 成功！共 1517 条记录
正在下载: 000625 长安汽车 (汽车)
[2026-04-10 10:41:54] SUCCESS  stock_000625 shape=(1517, 7)
  ✓ 成功！共 1517 条记录
正在下载: 601633 长城汽车 (汽车)
[2026-04-10 10:41:55] SUCCESS  stock_601633 shape=(1517, 7)
  ✓ 成功！共 1517 条记录
正在下载: 000002 万科A (房地产)
[2026-04-10 10:41:56] SUCCESS  stock_000002 shape=(1517, 7)
  ✓ 成功！共 1517 条记录
正在下载: 600519 贵州茅台 (白酒)
[2026-04-10 10:42:00] SUCCESS  stock_600519 shape=(1517, 7)
  ✓ 成功！共 1517 条记录
正在下载: 000858 五粮液 (白酒)
[2026-04-10 10:42:01] SUCCESS  stock_000858 shape=(1517, 7)
  ✓ 成功！共 1517 条记录
正在下载: 601857 中国石油 (能源)
[2026-04-10 10:42:03] SUCCESS  stock_601857 shape=(1517, 7)
  ✓ 成功！共 1517 条记录
正在下载: 600050 中国联通 (通讯)
[2026-04-10 10:42:04] SUCCESS  stock_600050 shape=(1517, 7)
  ✓ 成功！共 1517 条记录
正在下载: 002352 顺丰控股 (物流)
[2026-04-10 10:42:07] SUCCESS  stock_002352

##1.2 市场指数下载

In [3]:
# ========== 1.2 市场指数下载 ==========
print("="*60)
print("1.2 下载市场指数数据")
print("="*60)

# 创建指数文件夹
index_folder = os.path.join(data_folder, "index")
if not os.path.exists(index_folder):
    os.makedirs(index_folder)
    print(f"✓ 创建文件夹: {index_folder}/")

indices = [
    ("000300", "沪深300", "沪深300指数，A股市场基准，用于CAPM分析"),
    ("000905", "中证500", "中证500指数，代表中小盘股，与沪深300形成互补"),
]

bs.login()

for code, name, reason in indices:
    try:
        print(f"正在下载: {code} {name}")
        
        rs = bs.query_history_k_data_plus(
            f"sh.{code}",
            "date,open,high,low,close,volume,amount",
            start_date=start_date,
            end_date=end_date,
            frequency="d",
            adjustflag="1"
        )
        
        data_list = []
        while (rs.error_code == '0') and rs.next():
            data_list.append(rs.get_row_data())
        
        if data_list:
            df = pd.DataFrame(data_list,
                            columns=['日期', '开盘', '最高', '最低', '收盘', '成交量', '成交额'])
            for col in ['开盘', '最高', '最低', '收盘', '成交量', '成交额']:
                df[col] = pd.to_numeric(df[col])
            
            # 新文件名格式：index_代码_名称.csv
            filename = os.path.join(index_folder, f"index_{code}_{name}.csv")
            df.to_csv(filename, index=False, encoding="utf-8-sig")
            
            write_log("SUCCESS", "index", code, f"shape={df.shape}")
            print(f"  ✓ 成功！共 {len(df)} 条记录")
        else:
            raise Exception("No data returned")
        
        time.sleep(0.3)
        
    except Exception as e:
        write_log("FAILED", "index", code, str(e))
        print(f"  ✗ 失败: {e}")

bs.logout()
print("\n指数数据下载完成！")

1.2 下载市场指数数据


KeyboardInterrupt: 

##1.3 宏观经济指标（CPI + 汇率）

In [ ]:
# ========== 1.3 宏观经济指标下载（修复版）==========
print("="*60)
print("1.3 下载宏观经济指标")
print("="*60)

import akshare as ak
import pandas as pd
from datetime import datetime
import time
import os
import re

# 创建宏观数据文件夹
data_folder = "data"
macro_folder = os.path.join(data_folder, "macro")
if not os.path.exists(macro_folder):
    os.makedirs(macro_folder)
    print(f"✓ 创建文件夹: {macro_folder}/")

# 日志函数
def write_log(status, data_type, name, details=""):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open("download_log.txt", "a", encoding="utf-8") as f:
        if status == "SUCCESS":
            f.write(f"[{timestamp}] SUCCESS  {data_type}_{name} {details}\n")
        else:
            f.write(f"[{timestamp}] FAILED   {data_type}_{name} Error: {details}\n")

macro_results = []

# ---------- 1. CPI 同比增速（必选）----------
print("\n正在下载: CPI 同比增速")
print("  理由: CPI是衡量通胀的核心指标，影响货币政策走向，进而影响股市估值")

try:
    # 使用 macro_china_cpi_yearly 获取年度数据，或使用其他稳定接口
    # 方案1: 使用 macro_china_cpi_monthly 但需要正确解析
    df_cpi = ak.macro_china_cpi_monthly()
    
    print(f"  CPI数据列名: {df_cpi.columns.tolist()}")
    print(f"  前3行数据: \n{df_cpi.head(3)}")
    
    # 正确解析数据
    # 根据列名，'日期' 列包含 "2026年02月份" 格式
    if '日期' in df_cpi.columns:
        # 提取年月：将 "2026年02月份" 转换为 "2026-02"
        def parse_chinese_date(date_str):
            # 匹配 "2026年02月份" 格式
            match = re.search(r'(\d{4})年(\d{1,2})月份?', str(date_str))
            if match:
                year, month = match.groups()
                return f"{year}-{int(month):02d}"
            return date_str
        
        df_cpi['日期'] = df_cpi['日期'].astype(str).apply(parse_chinese_date)
        
        # 找到 CPI 同比列（通常包含"同比"）
        cpi_col = None
        for col in df_cpi.columns:
            if '同比' in col:
                cpi_col = col
                break
        
        if cpi_col is None:
            # 如果没有找到，使用第三列（通常是今值）
            cpi_col = df_cpi.columns[2] if len(df_cpi.columns) > 2 else df_cpi.columns[1]
        
        df_result = df_cpi[['日期', cpi_col]].copy()
        df_result.columns = ['日期', 'CPI同比(%)']
        
        # 筛选 2020 年之后的数据
        df_result = df_result[df_result['日期'] >= '2020-01']
        
        # 保存
        cpi_file = os.path.join(macro_folder, "macro_cpi.csv")
        df_result.to_csv(cpi_file, index=False, encoding="utf-8-sig")
        
        write_log("SUCCESS", "macro", "CPI", f"shape={df_result.shape}")
        print(f"  ✓ 成功！共 {len(df_result)} 条记录 → {cpi_file}")
        
    else:
        raise Exception("未找到日期列")

except Exception as e:
    print(f"  macro_china_cpi_monthly 解析失败: {e}")
    
    # 备用方案1：使用 macro_china_cpi
    try:
        print("  尝试备用接口 macro_china_cpi...")
        df_cpi = ak.macro_china_cpi()
        
        print(f"  列名: {df_cpi.columns.tolist()}")
        
        # 处理日期格式
        if '日期' in df_cpi.columns:
            # 提取年份
            df_cpi['日期'] = df_cpi['日期'].astype(str)
            # 匹配 "2026年02月份" 格式
            df_cpi['日期'] = df_cpi['日期'].str.extract(r'(\d{4})年')[0]
            
            # 找到 CPI 列
            cpi_col = '当月' if '当月' in df_cpi.columns else df_cpi.columns[1]
            
            df_result = df_cpi[['日期', cpi_col]].copy()
            df_result.columns = ['日期', 'CPI同比(%)']
            df_result = df_result[df_result['日期'] >= '2020']
            
            cpi_file = os.path.join(macro_folder, "macro_cpi.csv")
            df_result.to_csv(cpi_file, index=False, encoding="utf-8-sig")
            
            write_log("SUCCESS", "macro", "CPI", f"shape={df_result.shape}")
            print(f"  ✓ 成功！共 {len(df_result)} 条记录")
        else:
            raise Exception("未找到日期列")
            
    except Exception as e2:
        # 备用方案2：使用世界银行 API 获取中国 CPI
        print("  尝试备用接口 macro_china_cpi_yearly...")
        try:
            df_cpi = ak.macro_china_cpi_yearly()
            
            df_result = df_cpi.iloc[:, [0, 1]].copy()
            df_result.columns = ['年份', 'CPI同比(%)']
            df_result = df_result[df_result['年份'] >= 2020]
            df_result['年份'] = df_result['年份'].astype(str)
            
            cpi_file = os.path.join(macro_folder, "macro_cpi_annual.csv")
            df_result.to_csv(cpi_file, index=False, encoding="utf-8-sig")
            
            write_log("SUCCESS", "macro", "CPI", f"shape={df_result.shape} (年度数据)")
            print(f"  ✓ 成功！共 {len(df_result)} 条记录 (年度数据)")
            
        except Exception as e3:
            write_log("FAILED", "macro", "CPI", str(e3))
            print(f"  ✗ 所有接口都失败: {e3}")

time.sleep(1)

# ---------- 2. 自选指标：M2 货币供应量同比增速 ----------
print("\n正在下载: M2 货币供应量同比增速")
print("  理由: M2反映市场流动性，与股市资金面高度相关")

try:
    # 获取货币供应量数据
    # macro_china_money_supply 或 macro_china_supply_of_money
    df_m2 = ak.macro_china_supply_of_money()
    
    print(f"  M2数据列名: {df_m2.columns.tolist()}")
    
    # 查找 M2 相关的列
    m2_col = None
    date_col = None
    
    for col in df_m2.columns:
        if 'M2' in col or '广义货币' in col:
            if '同比' in col or '同比增长' in col:
                m2_col = col
        if '时间' in col or '月份' in col or '日期' in col:
            date_col = col
    
    if m2_col and date_col:
        df_result = df_m2[[date_col, m2_col]].copy()
        df_result.columns = ['日期', 'M2同比(%)']
    else:
        # 通用处理
        df_result = df_m2.iloc[:, [0, 1]].copy()
        df_result.columns = ['日期', 'M2同比(%)']
    
    # 处理日期格式
    df_result['日期'] = df_result['日期'].astype(str)
    df_result = df_result[df_result['日期'] >= '2020']
    
    # 保存
    m2_file = os.path.join(macro_folder, "macro_m2.csv")
    df_result.to_csv(m2_file, index=False, encoding="utf-8-sig")
    
    write_log("SUCCESS", "macro", "M2", f"shape={df_result.shape}")
    print(f"  ✓ 成功！共 {len(df_result)} 条记录 → {m2_file}")
    
except Exception as e:
    print(f"  macro_china_supply_of_money 失败: {e}")
    # 备用方案：尝试其他 M2 接口
    try:
        print("  尝试备用接口 macro_china_money_supply...")
        df_m2 = ak.macro_china_money_supply()
        df_result = df_m2.iloc[:, [0, 1]].copy()
        df_result.columns = ['日期', 'M2同比(%)']
        df_result['日期'] = df_result['日期'].astype(str)
        df_result = df_result[df_result['日期'] >= '2020']
        
        m2_file = os.path.join(macro_folder, "macro_m2.csv")
        df_result.to_csv(m2_file, index=False, encoding="utf-8-sig")
        
        write_log("SUCCESS", "macro", "M2", f"shape={df_result.shape} (备用接口)")
        print(f"  ✓ 成功！共 {len(df_result)} 条记录")
    except Exception as e2:
        write_log("FAILED", "macro", "M2", str(e2))
        print(f"  ✗ 失败: {e2}")

# ---------- 3.备用自选指标：LPR 利率数据----------
print("\n正在下载: LPR 利率（备用指标）")
print("  理由: LPR影响企业和居民融资成本，与股市估值相关")

try:
    df_lpr = ak.macro_china_lpr()
    
    print(f"  LPR数据列名: {df_lpr.columns.tolist()}")
    
    # 提取 LPR1Y 和 LPR5Y
    if 'TRADE_DATE' in df_lpr.columns:
        df_lpr_result = df_lpr[['TRADE_DATE', 'LPR1Y', 'LPR5Y']].copy()
        df_lpr_result.columns = ['日期', 'LPR_1Y(%)', 'LPR_5Y(%)']
    else:
        df_lpr_result = df_lpr.iloc[:, [0, 1, 2]].copy()
        df_lpr_result.columns = ['日期', 'LPR_1Y(%)', 'LPR_5Y(%)']
    
    # 筛选 2020 年之后的数据
    df_lpr_result['日期'] = pd.to_datetime(df_lpr_result['日期'])
    df_lpr_result = df_lpr_result[df_lpr_result['日期'] >= '2020-01-01']
    df_lpr_result['日期'] = df_lpr_result['日期'].dt.strftime('%Y-%m-%d')
    
    lpr_file = os.path.join(macro_folder, "macro_lpr.csv")
    df_lpr_result.to_csv(lpr_file, index=False, encoding="utf-8-sig")
    
    write_log("SUCCESS", "macro", "LPR", f"shape={df_lpr_result.shape}")
    print(f"  ✓ 成功！共 {len(df_lpr_result)} 条记录 → {lpr_file}")
    
except Exception as e:
    write_log("FAILED", "macro", "LPR", str(e))
    print(f"  ✗ 失败: {e}")

# 显示结果汇总
print("\n" + "="*60)
print("宏观指标下载完成！")
print("="*60)

# 列出生成的文件
print("\n生成的文件：")
for file in os.listdir(macro_folder):
    if file.endswith('.csv'):
        file_path = os.path.join(macro_folder, file)
        df_check = pd.read_csv(file_path)
        print(f"  📄 {file}: {len(df_check)} 条记录")

1.3 下载宏观经济指标

正在下载: CPI 同比增速
  理由: CPI是衡量通胀的核心指标，影响货币政策走向，进而影响股市估值
  CPI数据列名: ['商品', '日期', '今值', '预测值', '前值']
  前3行数据: 
          商品          日期   今值  预测值   前值
0  中国CPI月率报告  1996-02-01  2.1  NaN  NaN
1  中国CPI月率报告  1996-03-01  2.3  NaN  2.1
2  中国CPI月率报告  1996-04-01  0.6  NaN  2.3
  ✓ 成功！共 70 条记录 → data\macro\macro_cpi.csv

正在下载: M2 货币供应量同比增速
  理由: M2反映市场流动性，与股市资金面高度相关


  0%|          | 0/18 [00:00<?, ?it/s]

  M2数据列名: ['统计时间', '货币和准货币（广义货币M2）', '货币和准货币（广义货币M2）同比增长', '货币(狭义货币M1)', '货币(狭义货币M1)同比增长', '流通中现金(M0)', '流通中现金(M0)同比增长', '活期存款', '活期存款同比增长', '准货币', '准货币同比增长', '定期存款', '定期存款同比增长', '储蓄存款', '储蓄存款同比增长', '其他存款', '其他存款同比增长']
  ✓ 成功！共 74 条记录 → data\macro\macro_m2.csv

正在下载: LPR 利率（备用指标）
  理由: LPR影响企业和居民融资成本，与股市估值相关


  0%|          | 0/4 [00:00<?, ?it/s]

  LPR数据列名: ['TRADE_DATE', 'LPR1Y', 'LPR5Y', 'RATE_1', 'RATE_2']
  ✓ 成功！共 75 条记录 → data\macro\macro_lpr.csv

宏观指标下载完成！

生成的文件：
  📄 macro_cpi.csv: 70 条记录
  📄 macro_lpr.csv: 75 条记录
  📄 macro_m2.csv: 74 条记录


##1.4财务指标

In [ ]:
# ========== 1.4 财务指标获取 ==========
print("="*60)
print("1.4 获取财务指标 - 按季度（ROE + 净利润率）")
print("="*60)

# 创建财务数据文件夹
finance_folder = os.path.join(data_folder, "finance")
if not os.path.exists(finance_folder):
    os.makedirs(finance_folder)
    print(f"✓ 创建文件夹: {finance_folder}/")

stocks_financial = [
    ("600036", "招商银行"), ("601166", "兴业银行"),
    ("000625", "长安汽车"), ("601633", "长城汽车"),
    ("000002", "万科A"), ("600519", "贵州茅台"),
    ("000858", "五粮液"), ("601857", "中国石油"),
    ("600050", "中国联通"), ("002352", "顺丰控股"),
]

# 季度范围
quarters = []
for year in range(2020, 2026):
    for q in range(1, 5):
        quarters.append((year, q))

print(f"股票数量: {len(stocks_financial)} 只")
print(f"季度数量: {len(quarters)} 个")
print(f"总请求数: {len(stocks_financial) * len(quarters)} 次\n")

long_format_data = []

bs.login()

for idx, (code, name) in enumerate(stocks_financial):
    print(f"[{idx+1}/{len(stocks_financial)}] 正在获取: {code} {name}")
    
    bs_code = f"sh.{code}" if code.startswith('6') else f"sz.{code}"
    success_count = 0
    
    for year, quarter in quarters:
        try:
            time.sleep(0.1)
            
            rs = bs.query_profit_data(code=bs_code, year=year, quarter=quarter)
            
            roe = None
            net_profit_margin = None
            
            while rs.next():
                row = rs.get_row_data()
                if len(row) > 3 and row[3] and row[3] != '':
                    roe = float(row[3])
                if len(row) > 4 and row[4] and row[4] != '':
                    net_profit_margin = float(row[4])
            
            if roe is not None:
                long_format_data.append({
                    "code": code, "name": name, "year": year, "quarter": quarter,
                    "indicator": "ROE", "value": roe, "unit": "%"
                })
                success_count += 1
            
            if net_profit_margin is not None:
                long_format_data.append({
                    "code": code, "name": name, "year": year, "quarter": quarter,
                    "indicator": "net_profit_margin", "value": net_profit_margin, "unit": "%"
                })
                
        except Exception as e:
            pass
    
    print(f"  ✓ 获取到 {success_count} 条记录")
    write_log("SUCCESS", "finance", code, f"records={success_count}")
    time.sleep(0.5)

bs.logout()

# 保存财务数据
if long_format_data:
    df_long = pd.DataFrame(long_format_data)
    # 新文件名：finance_ratios.csv
    finance_file = os.path.join(finance_folder, "finance_ratios.csv")
    df_long.to_csv(finance_file, index=False, encoding="utf-8-sig")
    
    print(f"\n✅ 财务数据已保存: {finance_file}")
    print(f"总记录数: {len(df_long)} 条")
    print("\n数据预览：")
    print(df_long.head(10))
else:
    print("\n❌ 未获取到财务数据")

1.4 获取财务指标 - 按季度（ROE + 净利润率）
✓ 创建文件夹: data\finance/
股票数量: 10 只
季度数量: 24 个
总请求数: 240 次

login success!
[1/10] 正在获取: 600036 招商银行
  ✓ 获取到 24 条记录
[2026-04-09 15:56:11] SUCCESS  finance_600036 records=24
[2/10] 正在获取: 601166 兴业银行
  ✓ 获取到 24 条记录
[2026-04-09 15:56:42] SUCCESS  finance_601166 records=24
[3/10] 正在获取: 000625 长安汽车
  ✓ 获取到 23 条记录
[2026-04-09 15:57:16] SUCCESS  finance_000625 records=23
[4/10] 正在获取: 601633 长城汽车
  ✓ 获取到 24 条记录
[2026-04-09 15:58:08] SUCCESS  finance_601633 records=24
[5/10] 正在获取: 000002 万科A
  ✓ 获取到 24 条记录
[2026-04-09 15:59:00] SUCCESS  finance_000002 records=24
[6/10] 正在获取: 600519 贵州茅台
  ✓ 获取到 23 条记录
[2026-04-09 15:59:54] SUCCESS  finance_600519 records=23
[7/10] 正在获取: 000858 五粮液
  ✓ 获取到 23 条记录
[2026-04-09 16:00:35] SUCCESS  finance_000858 records=23
[8/10] 正在获取: 601857 中国石油
  ✓ 获取到 24 条记录
[2026-04-09 16:01:40] SUCCESS  finance_601857 records=24
[9/10] 正在获取: 600050 中国联通
  ✓ 获取到 24 条记录
[2026-04-09 16:02:33] SUCCESS  finance_600050 records=24
[10/10] 正在获取: 002352 顺丰控股
 

##1.5查看最终文件结构

In [ ]:
# ========== 查看生成的文件结构 ==========
print("="*60)
print("生成的文件结构")
print("="*60)

def list_files(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = '  ' * (level + 1)
        for file in files:
            if file not in ['.DS_Store', 'download_log.txt']:
                print(f'{subindent}{file}')

list_files('.')

# 显示日志内容
print("\n" + "="*60)
print("download_log.txt 内容预览：")
print("="*60)
if os.path.exists("download_log.txt"):
    with open("download_log.txt", "r", encoding="utf-8") as f:
        lines = f.readlines()
        for line in lines[-15:]:  # 显示最后15行
            print(line.strip())

生成的文件结构
./
  01_download.ipynb
  README.md
  data/
    finance/
      finance_ratios.csv
    index/
      index_000300_沪深300.csv
      index_000905_中证500.csv
    macro/
      macro_cpi.csv
      macro_lpr.csv
      macro_m2.csv
    stock/
      stock_000002_万科A.csv
      stock_000625_长安汽车.csv
      stock_000858_五粮液.csv
      stock_002352_顺丰控股.csv
      stock_600036_招商银行.csv
      stock_600050_中国联通.csv
      stock_600519_贵州茅台.csv
      stock_601166_兴业银行.csv
      stock_601633_长城汽车.csv
      stock_601857_中国石油.csv
  indices/
    中证500.csv
    沪深300.csv
  macro/
    USDCNY_汇率.csv

download_log.txt 内容预览：
[2026-04-09 15:53:07] SUCCESS  macro_CPI shape=(75, 2) (手动构建)
[2026-04-09 15:53:07] FAILED   macro_USD_CNY Error: name 'ak' is not defined
[2026-04-09 15:56:11] SUCCESS  finance_600036 records=24
[2026-04-09 15:56:42] SUCCESS  finance_601166 records=24
[2026-04-09 15:57:16] SUCCESS  finance_000625 records=23
[2026-04-09 15:58:08] SUCCESS  finance_601633 records=24
[2026-04-09 15:59:00] SUCC